# Step 3.1 — Scaling

**Goal:** Scale both feature matrices so that no single feature dominates distance calculations due to magnitude differences.

**Why scaling matters for clustering:**  
K-Means and DBSCAN both rely on Euclidean distance. If one feature dimension has a much larger range than others, it will dominate the distance — irrespective of its actual informativeness. Scaling puts all features on a comparable footing.

**Different strategies for our two matrices:**

| Matrix | Type | Scaler | Rationale |
|--------|------|--------|-----------|
| TF-IDF | sparse (733 × 5000) | `MaxAbsScaler` | Preserves sparsity; each feature scaled by its maximum absolute value. sklearn's `TfidfVectorizer` already applies L2 document-level normalization (`norm='l2'`), so feature-level variation is the remaining concern. `StandardScaler` would require `with_mean=False` and still densifies intermediate results. |
| Word2Vec | dense (733 × 100) | `StandardScaler` | Dense data allows full standardization (zero mean, unit variance per dimension). Ensures all 100 embedding dimensions contribute equally to distance calculations. |

## 1. Setup

In [ ]:
import numpy as np
import scipy.sparse as sp
import pandas as pd
from sklearn.preprocessing import MaxAbsScaler, StandardScaler
import matplotlib.pyplot as plt
import os

TFIDF_MATRIX_PATH    = "../data/processed/tfidf_matrix.npz"
W2V_MATRIX_PATH      = "../data/processed/word2vec_matrix.npy"
TFIDF_SCALED_PATH    = "../data/processed/tfidf_scaled.npz"
W2V_SCALED_PATH      = "../data/processed/word2vec_scaled.npy"

os.makedirs("../data/processed", exist_ok=True)

# load_npz returns COO format — convert to CSR for efficient row/column operations
X_tfidf = sp.load_npz(TFIDF_MATRIX_PATH).tocsr()
X_w2v   = np.load(W2V_MATRIX_PATH)

print(f"TF-IDF matrix:   {X_tfidf.shape}, format={X_tfidf.format}")
print(f"Word2Vec matrix: {X_w2v.shape}, dtype={X_w2v.dtype}")


## 2. Pre-scaling Statistics

In [ ]:
# Convert to dense only for statistics (733x5000 ≈ 30 MB — fine)
tfidf_dense = X_tfidf.toarray()
tfidf_max_per_feature  = tfidf_dense.max(axis=0)
tfidf_mean_per_feature = tfidf_dense.mean(axis=0)

print("=== TF-IDF (pre-scaling) ===")
print(f"Max value across all features:  {tfidf_max_per_feature.max():.4f}")
print(f"Mean of per-feature maxima:     {tfidf_max_per_feature.mean():.4f}")
print(f"Std of per-feature maxima:      {tfidf_max_per_feature.std():.4f}")

print("\n=== Word2Vec (pre-scaling) ===")
print(f"Per-dimension mean — min: {X_w2v.mean(axis=0).min():.4f}, max: {X_w2v.mean(axis=0).max():.4f}")
print(f"Per-dimension std  — min: {X_w2v.std(axis=0).min():.4f},  max: {X_w2v.std(axis=0).max():.4f}")


## 3. Scale TF-IDF — MaxAbsScaler

In [ ]:
scaler_tfidf = MaxAbsScaler()
X_tfidf_scaled = scaler_tfidf.fit_transform(X_tfidf)

max_after = X_tfidf_scaled.toarray().max(axis=0)
print("TF-IDF after MaxAbsScaler:")
print(f"  Shape:        {X_tfidf_scaled.shape}")
print(f"  Still sparse: {sp.issparse(X_tfidf_scaled)}")
print(f"  Max per feature — min: {max_after.min():.4f}, max: {max_after.max():.4f}  (all should be ≤ 1.0)")


## 4. Scale Word2Vec — StandardScaler

In [ ]:
scaler_w2v = StandardScaler()
X_w2v_scaled = scaler_w2v.fit_transform(X_w2v)

print("Word2Vec after StandardScaler:")
print(f"  Shape: {X_w2v_scaled.shape}")
print(f"  Per-dimension mean — min: {X_w2v_scaled.mean(axis=0).min():.6f}, max: {X_w2v_scaled.mean(axis=0).max():.6f}  (≈ 0)")
print(f"  Per-dimension std  — min: {X_w2v_scaled.std(axis=0).min():.4f},  max: {X_w2v_scaled.std(axis=0).max():.4f}  (≈ 1)")

## 5. Visualize Feature Variance Before / After

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# TF-IDF: non-zero value distribution before scaling
axes[0].hist(X_tfidf.data, bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("TF-IDF: non-zero values (before MaxAbsScaler)")
axes[0].set_xlabel("TF-IDF weight")
axes[0].set_ylabel("Count")

# Word2Vec: std per dimension before scaling
axes[1].hist(X_w2v.std(axis=0), bins=30, color="darkorange", edgecolor="white")
axes[1].set_title("Word2Vec: std per dimension (before StandardScaler)")
axes[1].set_xlabel("Std")
axes[1].set_ylabel("Count")

plt.suptitle("Feature distributions before scaling", fontsize=13)
plt.tight_layout()
plt.savefig("../report/figures/03_1_scaling.png", dpi=150)
plt.show()

# After-scaling summary (degenerate distributions — not useful as histograms)
print("After scaling:")
print(f"  TF-IDF (MaxAbsScaler):   all feature maxima = 1.0 by definition")
print(f"  Word2Vec (StandardScaler): mean≈0 per dim, std≈1 per dim")
print(f"  W2V scaled std range: {X_w2v_scaled.std(axis=0).min():.6f} – {X_w2v_scaled.std(axis=0).max():.6f}")
print("Figure saved.")


## 6. Persist

In [ ]:
sp.save_npz(TFIDF_SCALED_PATH, X_tfidf_scaled)
print(f"Saved scaled TF-IDF matrix to {TFIDF_SCALED_PATH}")

np.save(W2V_SCALED_PATH, X_w2v_scaled)
print(f"Saved scaled Word2Vec matrix to {W2V_SCALED_PATH}")

## 7. Summary

**What we did:**
- Applied `MaxAbsScaler` to the sparse TF-IDF matrix — preserves sparsity, scales each feature to [-1, 1] range
- Applied `StandardScaler` to the dense Word2Vec matrix — zero mean, unit variance per dimension
- Persisted scaled matrices as `tfidf_scaled.npz` and `word2vec_scaled.npy`

**Next step:** `03_2_outlier_detection.ipynb` — identify laws that are extreme outliers before clustering.